## PakMart Traders – Pre-processing for ML

Prepares the `fact_sale` data for machine learning — predicting **total sale amount (TotalIncludingTax)** from transaction features.

The target column is `TotalIncludingTax` (PKR).

In [ ]:
from pyspark.sql.functions import *

### Data Cleansing and Feature Engineering

In [ ]:
fact_sale    = spark.table('fact_sale')
dim_city     = spark.table('dimension_city')
dim_stock    = spark.table('dimension_stock_item')
dim_customer = spark.table('dimension_customer')
dim_date     = spark.table('dimension_date')

print(f'Raw fact_sale rows: {fact_sale.count():,}')

In [ ]:
# Join dimensions to enrich fact table
enriched_df = fact_sale \
    .join(dim_city.select('CityKey','StateProvince','SalesTerritory'), 'CityKey') \
    .join(dim_stock.select('StockItemKey','IsChillerStock'), 'StockItemKey') \
    .join(dim_customer.select('CustomerKey','Category','BuyingGroup'), 'CustomerKey') \
    .join(dim_date.select('Date','CalendarYear','CalendarMonthNumber','FiscalYear','FiscalMonthNumber'),
          fact_sale.InvoiceDateKey == dim_date.Date)

# Add derived features
prep_df = enriched_df \
    .withColumn('InvoiceHour',    hour(col('InvoiceDateKey'))) \
    .withColumn('DayOfWeek',      dayofweek(col('InvoiceDateKey'))) \
    .withColumn('DayOfWeekName',  date_format(col('InvoiceDateKey'), 'EEEE')) \
    .withColumn('IsWeekend',      (dayofweek(col('InvoiceDateKey')).isin([1, 7])).cast('integer')) \
    .withColumn('IsFriday',       (dayofweek(col('InvoiceDateKey')) == 6).cast('integer')) \
    .withColumn('IsChillerItem',  col('IsChillerStock').cast('integer')) \
    .withColumn('Season',
        when((col('CalendarMonthNumber').isin(3,4,5)),            'Spring') \
        .when((col('CalendarMonthNumber').isin(6,7,8)),           'Summer') \
        .when((col('CalendarMonthNumber').isin(9,10,11)),         'Autumn') \
        .otherwise('Winter'))

In [ ]:
# Filter out anomalies
clean_df = prep_df.filter(
    (col('TotalIncludingTax') > 0) &
    (col('Quantity') > 0) & (col('Quantity') <= 1000) &
    (col('UnitPrice') > 0) & (col('UnitPrice') <= 500000) &
    (col('Profit').cast('float') >= -50000)
)

print(f'Clean rows: {clean_df.count():,}')

In [ ]:
table_name = 'pakmart_sale_clean'
clean_df.write.mode('overwrite').format('delta').saveAsTable(table_name)
print(f'Saved to delta table: {table_name}')

In [ ]:
display(clean_df)